# Cancer Cell Detection — PyTorch
### Breast Cancer Wisconsin Dataset | Binary Classification (Malignant vs Benign)

This notebook builds a simple feedforward neural network in **PyTorch** to classify tumor samples as malignant or benign, based on 30 numeric features computed from digitized images of fine needle aspirate (FNA) biopsies.

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — built into scikit-learn, 569 samples, 30 features, 2 classes.

**Goal:** Demonstrate a complete, working PyTorch training pipeline — data loading, preprocessing, model definition, training loop, and evaluation.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Load and Explore the Dataset

The dataset contains features like mean radius, mean texture, mean perimeter, mean area, mean smoothness, etc., computed from cell nuclei in the biopsy images. Target: `0 = malignant`, `1 = benign`.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"\nClass distribution:\n{df['diagnosis'].value_counts()}")
df.head()

In [ ]:
# Quick visualization of class balance and a couple of key features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='diagnosis', ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution')

sns.scatterplot(data=df, x='mean radius', y='mean texture', hue='diagnosis', ax=axes[1], palette='Set2')
axes[1].set_title('Mean Radius vs Mean Texture')

plt.tight_layout()
plt.show()

## 3. Preprocess the Data

- Train/test split (80/20, stratified to preserve class balance)
- Standardize features (zero mean, unit variance) — important since features are on very different scales (e.g. area vs smoothness)
- Convert to PyTorch tensors

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(f"Train shape: {X_train_t.shape}, Test shape: {X_test_t.shape}")

## 4. Define the Model

A simple feedforward neural network:
`Input(30) -> Dense(32, ReLU) -> Dense(16, ReLU) -> Dense(1, Sigmoid)`

Sigmoid output gives a probability of being benign (class 1).

In [ ]:
class CancerNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = CancerNet(input_dim=X_train.shape[1])
print(model)

## 5. Train the Model

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
train_losses, val_losses = [], []

for epoch in range(epochs):
    # Training step
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # Validation step
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_t)
        val_loss = criterion(val_outputs, y_test_t)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

## 6. Plot Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.title('Training vs Validation Loss — PyTorch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    probs = model(X_test_t)
    preds = (probs >= 0.5).float()

acc = accuracy_score(y_test_t, preds)
print(f"Test Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test_t, preds, target_names=['malignant', 'benign']))

cm = confusion_matrix(y_test_t, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['malignant', 'benign'], yticklabels=['malignant', 'benign'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — PyTorch')
plt.show()

## 8. Summary

- Built a feedforward neural network in **PyTorch** for binary classification of breast cancer biopsy data.
- Achieved strong test accuracy with a simple 2-hidden-layer architecture.
- Pipeline covered: data loading → preprocessing (standardization) → model definition → manual training loop → evaluation (accuracy, confusion matrix, classification report).

**Next steps (optional extensions):**
- Try deeper architectures or dropout for regularization
- Use `DataLoader`/mini-batches instead of full-batch training
- Compare against the TensorFlow version of this same pipeline
- Try a real cancer cell **image** dataset (e.g. histopathology images) for a CNN-based version